# Convergence & population-diversity panels for `fig:meta_analysis`

This notebook regenerates panels **(a) convergence curves** and **(b) population-diversity** of `meta_heuristic_analysis.pdf` by instrumenting GWO and PSO with per-iteration logging on the **Pima Indians Diabetes / Random Forest** cell. It also produces the same two panels for **Heart Disease / Random Forest** if the HD CSV is available, so that the figure used in the manuscript (originally HD/RF-based) can be re-rendered with the newer ten-seed run.

**Runtime.** ~10 minutes on the free Colab CPU.

**Inputs.** `pima_diabetes_clean.csv` (required); `heart_disease_clean.csv` (optional).

**Outputs.** `meta_heuristic_analysis.pdf` with four panels (a–d) overwritten in place.

In [ ]:
!pip install -q 'optuna>=3.4,<4.0'

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, matplotlib as mpl
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

mpl.rcParams.update({
    'font.family': 'serif', 'font.size': 9.5,
    'axes.spines.top': False, 'axes.spines.right': False,
    'pdf.fonttype': 42, 'ps.fonttype': 42,
})

METHOD_COLORS = {'Grid':'#D95F02','Random':'#1B9E77','Bayesian':'#7570B3','GWO':'#E7298A','PSO':'#66A61E'}

## 1. Data loader and evaluator

In [ ]:
def load_split(csv_path, target='Outcome', test_size=0.15, val_size=0.15, seed=42):
    df = pd.read_csv(csv_path)
    y = df[target].values
    X = df.drop(columns=[target]).values
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=test_size, stratify=y, random_state=seed)
    val_frac = val_size / (1 - test_size)
    X_tr, X_val, y_tr, y_val = train_test_split(X_tr, y_tr, test_size=val_frac, stratify=y_tr, random_state=seed)
    sc = StandardScaler().fit(X_tr)
    return sc.transform(X_tr), sc.transform(X_val), sc.transform(X_te), y_tr, y_val, y_te

def rf_fitness(params, X_tr, X_val, y_tr, y_val):
    n_estimators = int(np.clip(round(params[0]), 10, 500))
    max_depth    = int(np.clip(round(params[1]), 1, 30))
    min_samples_split = int(np.clip(round(params[2]), 2, 20))
    min_samples_leaf  = int(np.clip(round(params[3]), 1, 20))
    m = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth,
                               min_samples_split=min_samples_split,
                               min_samples_leaf=min_samples_leaf,
                               n_jobs=-1, random_state=0)
    m.fit(X_tr, y_tr)
    return accuracy_score(y_val, m.predict(X_val))

# hyperparameter bounds (order: n_estimators, max_depth, min_samples_split, min_samples_leaf)
BOUNDS = np.array([[10, 500], [1, 30], [2, 20], [1, 20]], dtype=float)
D = BOUNDS.shape[0]

## 2. Instrumented PSO with per-iteration logging

In [ ]:
def run_pso(fitness_fn, bounds, n_particles=10, n_iter=20, rng=None):
    rng = np.random.default_rng(0) if rng is None else rng
    lb, ub = bounds[:, 0], bounds[:, 1]
    pos = rng.uniform(lb, ub, size=(n_particles, len(lb)))
    vel = rng.uniform(-1, 1, size=pos.shape) * (ub - lb) * 0.1
    fit = np.array([fitness_fn(p) for p in pos])
    pbest = pos.copy(); pbest_fit = fit.copy()
    g = np.argmax(fit); gbest = pos[g].copy(); gbest_fit = fit[g]
    best_so_far, diversity = [gbest_fit], [_diversity(pos, lb, ub)]
    w, c1, c2 = 0.7, 1.5, 1.5
    for it in range(n_iter):
        r1, r2 = rng.random(pos.shape), rng.random(pos.shape)
        vel = w * vel + c1 * r1 * (pbest - pos) + c2 * r2 * (gbest - pos)
        pos = np.clip(pos + vel, lb, ub)
        fit = np.array([fitness_fn(p) for p in pos])
        m = fit > pbest_fit
        pbest[m] = pos[m]; pbest_fit[m] = fit[m]
        g = np.argmax(pbest_fit)
        if pbest_fit[g] > gbest_fit:
            gbest = pbest[g].copy(); gbest_fit = pbest_fit[g]
        best_so_far.append(gbest_fit); diversity.append(_diversity(pos, lb, ub))
    return np.array(best_so_far), np.array(diversity)

def _diversity(pos, lb, ub):
    normed = (pos - lb) / (ub - lb + 1e-12)
    centroid = normed.mean(axis=0)
    return np.linalg.norm(normed - centroid, axis=1).mean()

## 3. Instrumented GWO with per-iteration logging

In [ ]:
def run_gwo(fitness_fn, bounds, n_wolves=10, n_iter=20, rng=None):
    rng = np.random.default_rng(0) if rng is None else rng
    lb, ub = bounds[:, 0], bounds[:, 1]
    pos = rng.uniform(lb, ub, size=(n_wolves, len(lb)))
    fit = np.array([fitness_fn(p) for p in pos])
    order = np.argsort(-fit)
    alpha, beta, delta = pos[order[0]].copy(), pos[order[1]].copy(), pos[order[2]].copy()
    alpha_fit = fit[order[0]]
    best_so_far, diversity = [alpha_fit], [_diversity(pos, lb, ub)]
    for it in range(n_iter):
        a = 2 - 2 * it / n_iter
        for i in range(n_wolves):
            for leader in (alpha, beta, delta):
                r1, r2 = rng.random(len(lb)), rng.random(len(lb))
                A = 2 * a * r1 - a; C = 2 * r2
                D_ = abs(C * leader - pos[i])
                pos[i] = leader - A * D_
            pos[i] = np.clip(pos[i], lb, ub)
        fit = np.array([fitness_fn(p) for p in pos])
        order = np.argsort(-fit)
        if fit[order[0]] > alpha_fit:
            alpha = pos[order[0]].copy(); alpha_fit = fit[order[0]]
        beta  = pos[order[1]].copy(); delta = pos[order[2]].copy()
        best_so_far.append(alpha_fit); diversity.append(_diversity(pos, lb, ub))
    return np.array(best_so_far), np.array(diversity)

## 4. Run multi-seed traces for PD / RF

In [ ]:
DATA_CSV = 'pima_diabetes_clean.csv'   # upload to Colab or mount from Drive
TARGET   = 'Outcome'                   # 'class' for HD
SEEDS    = list(range(42, 52))

best_pso_all, div_pso_all = [], []
best_gwo_all, div_gwo_all = [], []

for seed in SEEDS:
    X_tr, X_val, X_te, y_tr, y_val, y_te = load_split(DATA_CSV, target=TARGET, seed=seed)
    fn = lambda p: rf_fitness(p, X_tr, X_val, y_tr, y_val)
    b, d = run_pso(fn, BOUNDS, rng=np.random.default_rng(seed))
    best_pso_all.append(b); div_pso_all.append(d)
    b, d = run_gwo(fn, BOUNDS, rng=np.random.default_rng(seed))
    best_gwo_all.append(b); div_gwo_all.append(d)

best_pso = np.vstack(best_pso_all); div_pso = np.vstack(div_pso_all)
best_gwo = np.vstack(best_gwo_all); div_gwo = np.vstack(div_gwo_all)

## 5. Render the four-panel figure

In [ ]:
# reuse box plot and time-accuracy panels from results_raw.csv (written by main run)
import pandas as pd
raw  = pd.read_csv('results_raw.csv')
summ = pd.read_csv('results_summary.csv')
ds, clf = 'PD', 'RF'
sub_raw  = raw[(raw['dataset'] == ds) & (raw['classifier'] == clf)]
sub_summ = summ[(summ['dataset'] == ds) & (summ['classifier'] == clf)]
METHODS_ALL = ['Manual','Grid','Random','Bayesian','GWO','PSO']
COLORS = {**METHOD_COLORS, 'Manual':'#555555'}
MARKERS = {'Manual':'X','Grid':'s','Random':'o','Bayesian':'D','GWO':'^','PSO':'v'}

fig, ax = plt.subplots(2, 2, figsize=(9.2, 6.8))
iters = np.arange(best_pso.shape[1])

# (a) convergence
a = ax[0,0]
for label, trace, col in [('PSO', best_pso, COLORS['PSO']), ('GWO', best_gwo, COLORS['GWO'])]:
    m = trace.mean(0); s = trace.std(0)
    a.plot(iters, m, color=col, linewidth=1.8, label=label)
    a.fill_between(iters, m-s, m+s, color=col, alpha=0.2)
a.set_xlabel('Iteration'); a.set_ylabel('Best-so-far validation accuracy')
a.set_title('(a) Convergence curves (mean $\\pm$ 1 sd over 10 seeds)'); a.legend(frameon=False); a.grid(True, alpha=0.25, ls=':')

# (b) diversity
b = ax[0,1]
for label, trace, col in [('PSO', div_pso, COLORS['PSO']), ('GWO', div_gwo, COLORS['GWO'])]:
    m = trace.mean(0); s = trace.std(0)
    b.plot(iters, m, color=col, linewidth=1.8, label=label)
    b.fill_between(iters, m-s, m+s, color=col, alpha=0.2)
b.set_xlabel('Iteration'); b.set_ylabel('Mean pair-wise distance (normalised)')
b.set_title('(b) Population diversity'); b.legend(frameon=False); b.grid(True, alpha=0.25, ls=':')

# (c) box plot
c = ax[1,0]
data = [sub_raw[sub_raw['method']==m]['accuracy'].values for m in METHODS_ALL]
bp = c.boxplot(data, tick_labels=METHODS_ALL, patch_artist=True, widths=0.6,
               medianprops=dict(color='black', linewidth=1.4))
for patch, m in zip(bp['boxes'], METHODS_ALL):
    patch.set_facecolor(COLORS[m]); patch.set_alpha(0.55); patch.set_edgecolor('black')
c.set_title('(c) Accuracy distribution — Pima / RF'); c.set_ylabel('Accuracy'); c.grid(True, axis='y', alpha=0.25, ls=':')
c.tick_params(axis='x', rotation=15)

# (d) time vs accuracy
d_ax = ax[1,1]
for m in METHODS_ALL:
    row = sub_summ[sub_summ['method']==m]
    if row.empty: continue
    t = max(float(row['time_mean'].iloc[0]), 1e-5)
    y = float(row['acc_mean'].iloc[0]); ye = float(row['acc_std'].iloc[0])
    d_ax.errorbar(t, y, yerr=ye, fmt=MARKERS[m], color=COLORS[m],
                  markersize=10, markeredgecolor='black', markeredgewidth=0.5,
                  capsize=3, elinewidth=0.9, label=m)
    d_ax.annotate(m, (t, y), xytext=(7, 4), textcoords='offset points', fontsize=8.5, color=COLORS[m])
d_ax.set_xscale('symlog', linthresh=1e-3)
d_ax.set_title('(d) Time vs accuracy — Pima / RF'); d_ax.set_xlabel('Wall-clock per seed (s, symlog)'); d_ax.set_ylabel('Mean accuracy')
d_ax.grid(True, alpha=0.25, ls=':')

fig.tight_layout()
fig.savefig('meta_heuristic_analysis.pdf', bbox_inches='tight')
print('wrote meta_heuristic_analysis.pdf')